In [1]:
IN_DIR = "/content/outputs/m2_consensus"  # adjustable
ZIP_PATH = "/content/m2_consensus.zip"  # adjustable
RUN_DIR = "/content/outputs/m2_consensus_llm"  # adjustable
METHOD = "m2_consensus"
DRY_RUN = False  # adjustable

MODEL = "models/gemini-3.5-flash"  # adjustable
TEMPERATURE = 1.0  # adjustable
MAX_OUTPUT_TOKENS = 8192
N_RUNS = 5  # adjustable
TEMPLATES = ["T1", "T2", "T3", "T4"]  # adjustable
SLEEP_RANGE = (1, 5)  # adjustable
MAX_RETRIES = 6  # adjustable
REQUEST_TIMEOUT = 180

LETTER_SEED = 7
TOP_N_GENES = 10
TISSUE_CONTEXT = "human tumour tissue profiled with NanoString CosMx SMI, 960-gene panel"  # adjustable

CFG = {k: v for k, v in dict(globals()).items() if k.isupper()}

In [2]:
import os, re, json, time, random, hashlib, zipfile, glob, shutil, string
import pandas as pd, requests

RAW_DIR = f"{RUN_DIR}/raw"
os.makedirs(RAW_DIR, exist_ok=True)
NEED = "niche_markers_long.csv"
if not os.path.exists(f"{IN_DIR}/{NEED}"):
    zp = ZIP_PATH
    if not zp:
        from google.colab import files
        zp = next(iter(files.upload()))
    os.makedirs(IN_DIR, exist_ok=True)
    zipfile.ZipFile(zp).extractall(IN_DIR)
    hits = glob.glob(f"{IN_DIR}/**/{NEED}", recursive=True)
    assert hits, f"{NEED} not found after unzipping"
    IN_DIR = os.path.dirname(hits[0])

mk = pd.read_csv(f"{IN_DIR}/niche_markers_long.csv", dtype={"niche": str})
summ = pd.read_csv(f"{IN_DIR}/niche_summary.csv", dtype={"niche": str})
stab = pd.read_csv(f"{IN_DIR}/niche_stability.csv", dtype={"niche": str})
N_PARTITIONS = len(pd.read_csv(f"{IN_DIR}/base_partitions.csv", nrows=0).columns) - 1
EXPERT_NAMES = list(pd.read_csv(f"{IN_DIR}/crosstab_vs_region.csv", index_col=0, nrows=0).columns)

niche_ids = sorted(summ["niche"], key=int)
assert set(niche_ids) == set(mk["niche"]) == set(stab["niche"]), "niche ids differ across input files"
shuffled = niche_ids[:]
random.Random(LETTER_SEED).shuffle(shuffled)
letters = list(string.ascii_uppercase[:len(shuffled)])
letter_to_niche = dict(zip(letters, shuffled))
S = summ.set_index("niche"); ST = stab.set_index("niche")
key = pd.DataFrame({"niche_letter": letters, "niche_id": shuffled, "n_cells": [int(S.loc[x, "n_cells"]) for x in shuffled]})
key.to_csv(f"{RUN_DIR}/niche_letter_key.csv", index=False)
print(f"{len(letters)} niches (A-{letters[-1]}), {N_PARTITIONS} base partitions, input: {IN_DIR}")

8 niches (A-H), 9 base partitions, input: /content/outputs/m2_consensus


In [3]:
FORMAT = '[{"region": "A", "label": "...", "confidence": "high|medium|low", "justification": "..."}]'
HEAD = "You are an expert computational biologist and spatial transcriptomics specialist.\n\n### Task:\nIdentify and annotate the biological cellular niche (tissue microenvironment identity) for each region below.\n\n"
STAB_NOTE = f"agreement across {N_PARTITIONS} clustering runs, 0 to 1"

def gene_line(niche, with_fc):
    g = mk[mk["niche"] == niche].sort_values("rank").head(TOP_N_GENES)
    if with_fc: return ", ".join(f"{r.gene} (log2FC {r.log2FC:.2f})" for r in g.itertuples())
    return ", ".join(g["gene"])

def region_block(letter, tpl):
    nid = letter_to_niche[letter]
    lines = [f"region {letter}:", f"  Top {TOP_N_GENES} marker genes (ranked): {gene_line(nid, tpl in ('T3', 'T4'))}"]
    if tpl != "T1":
        lines.append(f"  Cells: {int(S.loc[nid, 'n_cells'])} ({100 * S.loc[nid, 'frac_cells']:.1f}% of all cells)")
        lines.append(f"  Stability ({STAB_NOTE}): {ST.loc[nid, 'mean_stability']:.2f}")
    return "\n".join(lines)

def build_prompt(tpl):
    meta = ["- Modality/Platform: NanoString CosMx SMI (960-gene panel)", f"- Number of regions: {len(letters)} (A-{letters[-1]})", "- Target Species: Human"]
    if tpl != "T1": meta.append(f"- Tissue context: {TISSUE_CONTEXT}")
    table = "\n\n".join(region_block(L, tpl) for L in letters)
    spec = f"a fenced ```json code block containing a JSON array with exactly one object per region, in this form:\n{FORMAT}\nUse a short biological niche label."
    if tpl == "T4":
        resp = "Think step by step before answering. For each region: (1) list the likely cell types, (2) state the evidence for each, (3) give one niche label. Then end your reply with " + spec
    else:
        resp = "Return only " + spec
    return HEAD + "### Metadata:\n" + "\n".join(meta) + "\n\n### Marker genes per region:\n" + table + "\n\n### Response format:\n" + resp

PROMPTS = {t: build_prompt(t) for t in TEMPLATES}
BANNED = {"banksy"} | {x.lower() for x in EXPERT_NAMES} | {"myeloid-enriched stroma", "stroma", "tumor interior", "lymphoid structure", "tumor-stroma boundary", "neutrophils", "immune", "macrophages"}
for t, p in PROMPTS.items():
    assert "Region" not in p, f"{t}: contains 'Region'"
    bad = [b for b in BANNED if b in p.lower()]
    assert not bad, f"{t}: contains banned terms {bad}"
    open(f"{RUN_DIR}/prompt_{t}.txt", "w").write(p)
if DRY_RUN:
    for t, p in PROMPTS.items(): print(f"\n{'=' * 20} {t} {'=' * 20}\n{p}")

In [4]:
class QuotaStop(Exception):
    pass

def call_gemini(prompt, api_key):
    url = f"https://generativelanguage.googleapis.com/v1/{MODEL}:generateContent"
    body = {"contents": [{"parts": [{"text": prompt}]}], "generationConfig": {"temperature": TEMPERATURE, "maxOutputTokens": MAX_OUTPUT_TOKENS}}
    for a in range(MAX_RETRIES):
        try:
            r = requests.post(url, headers={"x-goog-api-key": api_key, "Content-Type": "application/json"}, json=body, timeout=REQUEST_TIMEOUT)
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            print("connection error, retrying:", repr(e)[:100]); time.sleep(min(60, 5 * 2 ** a)); continue
        if r.status_code == 429: raise QuotaStop(r.text[:300])
        if r.status_code in (500, 503):
            print("HTTP", r.status_code, "retrying"); time.sleep(min(60, 5 * 2 ** a)); continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError("retries exhausted")

def extract(data):
    c = (data.get("candidates") or [{}])[0]
    parts = (c.get("content") or {}).get("parts") or []
    text = "".join(p.get("text", "") for p in parts if not p.get("thought"))
    return text, c.get("finishReason"), data.get("modelVersion")

def raw_path(t, r):
    return f"{RAW_DIR}/{t}_run{r}.json"

if DRY_RUN:
    print("DRY_RUN=True: no API calls made")
else:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY_3")
    todo = [(t, r) for r in range(1, N_RUNS + 1) for t in TEMPLATES if not os.path.exists(raw_path(t, r))]
    print(f"{len(todo)} calls remaining")
    for i, (t, r) in enumerate(todo):
        try:
            data = call_gemini(PROMPTS[t], api_key)
        except QuotaStop as e:
            print(f"429 quota reached; stopped. Saved so far; remaining: {todo[i:]}\n{e}"); break
        text, fin, ver = extract(data)
        json.dump({"template": t, "run": r, "model": MODEL, "model_version": ver, "temperature": TEMPERATURE, "finish_reason": fin,
                   "text": text, "usage": data.get("usageMetadata")}, open(raw_path(t, r), "w"), indent=1)
        print(f"saved {t} run {r} ({fin})")
        time.sleep(random.uniform(*SLEEP_RANGE))

20 calls remaining
saved T1 run 1 (STOP)
saved T2 run 1 (STOP)
saved T3 run 1 (STOP)
saved T4 run 1 (STOP)
saved T1 run 2 (STOP)
saved T2 run 2 (STOP)
saved T3 run 2 (STOP)
saved T4 run 2 (STOP)
saved T1 run 3 (STOP)
saved T2 run 3 (STOP)
saved T3 run 3 (STOP)
saved T4 run 3 (STOP)
saved T1 run 4 (STOP)
saved T2 run 4 (STOP)
saved T3 run 4 (STOP)
saved T4 run 4 (STOP)
saved T1 run 5 (STOP)
saved T2 run 5 (STOP)
saved T3 run 5 (STOP)
saved T4 run 5 (STOP)


In [5]:
FENCE = re.compile(r"```(?:json)?\s*(\[.*?\])\s*```", re.S)

def parse_reply(text):
    blocks = FENCE.findall(text or "")
    if not blocks: return None
    try: arr = json.loads(blocks[-1])
    except Exception: return None
    if not isinstance(arr, list) or not all(isinstance(o, dict) for o in arr): return None
    out = {str(o.get("region", "")).strip().upper(): o for o in arr}
    return out if set(letters) <= set(out) else None

COLS = ["method", "template", "run", "model", "model_version", "niche_letter", "niche_id", "n_cells", "label", "confidence", "justification", "parse_ok", "finish_reason", "raw_file"]
if DRY_RUN:
    print("DRY_RUN=True: nothing to collect")
else:
    rows, versions = [], set()
    for t in TEMPLATES:
        for r in range(1, N_RUNS + 1):
            if not os.path.exists(raw_path(t, r)): continue
            raw = json.load(open(raw_path(t, r)))
            versions.add(raw["model_version"])
            parsed = parse_reply(raw["text"])
            for L in letters:
                o = parsed[L] if parsed else {}
                rows.append({"method": METHOD, "template": t, "run": r, "model": raw["model"], "model_version": raw["model_version"], "niche_letter": L,
                             "niche_id": letter_to_niche[L], "n_cells": int(S.loc[letter_to_niche[L], "n_cells"]), "label": o.get("label"),
                             "confidence": o.get("confidence"), "justification": o.get("justification"), "parse_ok": parsed is not None,
                             "finish_reason": raw["finish_reason"], "raw_file": raw_path(t, r)})
    res = pd.DataFrame(rows, columns=COLS)
    res.to_csv(f"{RUN_DIR}/results_{METHOD}.csv", index=False)
    cfg = {"CONFIG": CFG, "in_dir_used": IN_DIR, "n_partitions": N_PARTITIONS, "temperature": TEMPERATURE, "model": MODEL,
           "model_versions": sorted(v for v in versions if v), "letter_to_niche": letter_to_niche,
           "prompt_sha256": {t: hashlib.sha256(p.encode()).hexdigest() for t, p in PROMPTS.items()},
           "n_raw_files": len(glob.glob(f"{RAW_DIR}/*.json")), "parse_rate": res.groupby("template")["parse_ok"].mean().to_dict() if len(res) else {}}
    json.dump(cfg, open(f"{RUN_DIR}/run_config.json", "w"), indent=2, default=str)
    print(len(res), "rows; parse rate:", cfg["parse_rate"])

160 rows; parse rate: {'T1': 1.0, 'T2': 1.0, 'T3': 1.0, 'T4': 1.0}


In [6]:
zp = shutil.make_archive(RUN_DIR, "zip", RUN_DIR)
from google.colab import files
files.download(zp)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>